# Reshape data

Each month's raw parquet stores one row per **(declaration item, tax concept)** pair: every shipment-level field (importer, quantity, FOB, countries, etc.) repeats identically across all tax-concept (`COD`) rows belonging to the same declaration, with only `MONTO` varying. Across 2019-2026 this inflates roughly 43M declaration items into 672.8M rows.

This step reshapes each month to one row per declaration item, summing `MONTO` across all tax-concept rows into a single `MONTO_TRIBUTADO_TOTAL` column, written next to the month's raw parquet as `impo_<yyyymm>_reshaped.parquet`. The per-concept breakdown (`COD`) is not kept.

Reshaping runs before decoding (0.4) on purpose: the AFIP code columns (`ADU`, `M`, `UN`, `DIV`, `PAI`, `PAI_duplicated_0`) are still raw here, but decoding is a deterministic code -> description mapping, so grouping on the raw codes gives the same groups as grouping on their decoded values. Doing it in this order means 0.4's catalog joins only run once per declaration item.

Working month by month keeps memory low and gives the same result as grouping the whole dataset at once: `FECHA_` (the month) is part of the group key and each monthly zip only contains its own month, so a declaration item's rows never span two months. Months already reshaped are skipped.

In [ ]:
import logging

from tqdm.auto import tqdm

from argentina_afip_comex.pipeline import INTERIM_DIR, reshape_month

logging.basicConfig(level=logging.INFO, format="%(message)s")

interim_paths = sorted(INTERIM_DIR.rglob("impo_??????.parquet"))
for interim_path in tqdm(interim_paths, desc="Reshaping months"):
    reshape_month(interim_path)